# Data Preprocessing

Preparing the NSL-KDD dataset for machine learning models.

## Objective

In this notebook we prepare the dataset for machine learning by:

- Loading the dataset
- Creating binary labels
- Removing unnecessary features
- Separating features and target
- Identifying categorical features
- Applying One-Hot Encoding
- Building a reusable preprocessing pipeline

In [10]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [11]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files",
    "num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate",
    "srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

train = pd.read_csv(
    "../data/raw/KDDTrain+.txt",
    names=columns
)

test = pd.read_csv(
    "../data/raw/KDDTest+.txt",
    names=columns
)

print(train.head())
print(test.head())
print(train.shape)


   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                    0.60                         0.88   
2             

In [12]:
train["binary_label"] = train["label"].apply(lambda x: 0 if x == "normal" else 1)

test["binary_label"] = test["label"].apply(lambda x: 0 if x == "normal" else 1)

In [13]:
X_train = train.drop(columns=["label", "binary_label", "difficulty"])
X_test = test.drop(columns=["label", "binary_label", "difficulty"])

y_train = train["binary_label"]
y_test = test["binary_label"]

In [14]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns

categorical_cols

/var/folders/4s/c_3w3vw96s53p_ffdszr4f3h0000gn/T/ipykernel_42531/3179385744.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns


Index(['protocol_type', 'service', 'flag'], dtype='str')

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols,
        )
    ],
    remainder="passthrough",
)

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(125973, 122)
(22544, 122)


In [17]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.pkl")

['../models/preprocessor.pkl']

# End od notbook 2

* Three categorical features were identified.
* One-Hot Encoding was applied.
* Unknown categories are ignored during testing.
* Difficulty level was removed because it is metadata.
* Final feature count increased from 41 to 123.